In [23]:
# Import necessary libraries and create paths to the Sentinel-2 bands
import rasterio
from pathlib import Path
from opencropphenotyping.io import find_band
from rasterio import transform
from rasterio.windows import Window

project_root = Path.cwd().parent

safe_dir = (project_root / "data" / "raw" / "sentinel_2" / "S2A_MSIL2A_20250804T104701_N0511_R051_T31TCJ_20250804T161517.SAFE")
output_dir = project_root / "data" / "toy dataset"

b04_path = find_band(safe_dir, "B04")
b08_path = find_band(safe_dir, "B08")

In [53]:
# Define a window to read a subset of the image
window = Window(
    col_off=4000,
    row_off=6000,
    width=512,
    height=512,
)
print(window)

Window(col_off=4000, row_off=6000, width=512, height=512)


In [54]:
with rasterio.open(b04_path) as src_b04, rasterio.open(b08_path) as src_b08:
    print(src_b04.profile)
    print(src_b08.profile)

    toy_b04 = src_b04.read(
        1,
        window=window,
    )
    toy_b08 = src_b08.read(
        1,
        window=window,
    )

{'driver': 'JP2OpenJPEG', 'dtype': 'uint16', 'nodata': None, 'width': 10980, 'height': 10980, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / UTM zone 31N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",3],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32631"]]'), 'transform': Affine(10.0, 0.0, 300000.0,
       0.0, -10.0, 4900020.0), 'blockxsize': 1024, 'blockysize': 1024, 'tiled': True}
{'driver': 'JP2OpenJPEG', 'dtype': 'uint16', 'nodata': None, 'width': 10980, 'height': 10980, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / UT

In [55]:
# Update the transform for the windowed data
transform = src_b04.window_transform(window)

# update the profile
profile_b04 = src_b04.profile.copy()
profile_b08 = src_b08.profile.copy()

profile_b04.update(
    width=window.width,
    height=window.height,
    transform=transform,
)
profile_b08.update(
    width=window.width,
    height=window.height,
    transform=transform,
)

In [56]:
# Write the toy image to a new file
with rasterio.open(output_dir / "toy_image_b04.tif", "w", **profile_b04) as dst:
    dst.write(toy_b04, 1)
with rasterio.open(output_dir / "toy_image_b08.tif", "w", **profile_b08) as dst:
    dst.write(toy_b08, 1)